# δ-Mem Compound-Stack Test Matrix

Cross-base retrofit study — Qwen3-4B-Instruct-2507 (primary) + SmolLM3-3B (secondary, adapter trained in pre-cell T1).

Tests δ-Mem, sliding-window attention, StreamingLLM, speculative decoding, and native MTP as compound levers over a vanilla full-attention baseline.

**Run on Kaggle:** `File → Import Notebook → URL` with the GitHub raw URL.  
**Stop a run:** push `control/STATUS=stop` to the repo from anywhere; the next cell boundary will exit gracefully.  
**Auto-stop:** wall-clock budget (default 8 h) or natural completion.

> Full spec in [`docs/spec.md`](../docs/spec.md). Scaffold version — cell dispatch is stubbed; runs report `status: scaffold-stub`.

## 1. Environment detection and repo sync

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, textwrap

REPO_URL = "https://github.com/jamesburton/delta-mem-smollm3-3b.git"
REPO_NAME = "delta-mem-smollm3-3b"

def detect_env():
    if pathlib.Path("/kaggle").exists():
        return "kaggle"
    if pathlib.Path("/content").exists():
        return "colab"
    return "local"

ENV = detect_env()
print("environment:", ENV)

if ENV == "kaggle":
    WORKDIR = pathlib.Path("/kaggle/working") / REPO_NAME
elif ENV == "colab":
    WORKDIR = pathlib.Path("/content") / REPO_NAME
else:
    # local — assume we're already in the repo
    WORKDIR = pathlib.Path.cwd()
    if WORKDIR.name != REPO_NAME and not (WORKDIR / "harness").exists():
        raise SystemExit("Run this notebook from the repo root locally, or on Kaggle/Colab.")

if ENV in ("kaggle", "colab") and not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
elif ENV in ("kaggle", "colab"):
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--rebase"], check=True)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("workdir:", WORKDIR)

## 2. Install dependencies

In [ ]:
if ENV in ("kaggle", "colab"):
    subprocess.run(["bash", "scripts/kaggle_bootstrap.sh"], check=True)
else:
    print("Local run — assuming requirements already installed.")

## 3. Verify GPU and imports

In [ ]:
import torch, transformers, accelerate
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB  CC {p.major}.{p.minor}")
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

from harness import state, control, cells, oracle
print("harness version:", __import__('harness').__version__)

## 4. Oracle authentication

Auto-detects: `claude` CLI (subscription) → `ANTHROPIC_API_KEY` (Python SDK) → unavailable.

On Kaggle, the cleanest path is to either:
1. install Claude Code CLI in this kernel and run `!claude login` interactively (device-flow), **or**
2. set `ANTHROPIC_API_KEY` as a Kaggle secret (Add-ons → Secrets) and the harness will pick it up automatically.

If neither, runs proceed without the frontier-ceiling row in the summary.

In [ ]:
# Optionally pull ANTHROPIC_API_KEY from Kaggle secrets into env
if ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try:
            os.environ["ANTHROPIC_API_KEY"] = us.get_secret("ANTHROPIC_API_KEY")
            print("loaded ANTHROPIC_API_KEY from Kaggle secrets")
        except Exception as e:
            print("no ANTHROPIC_API_KEY secret:", e)
    except ImportError:
        pass

auth_mode = oracle.detect_auth()
print("oracle auth mode:", auth_mode)
if auth_mode == "unavailable":
    print("⚠️  Runs will record `oracle: unavailable` and skip the ceiling row.")

## 5. Git push auth (for committing per-cell results)

Kaggle: set `GH_PAT_DELTA_MEM_TESTS` as a secret (a PAT with `repo` scope on this repository only). The notebook configures git to use it for pushes.

Local: assumes you already have `gh auth` or a pushable remote configured.

In [ ]:
if ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GH_PAT_DELTA_MEM_TESTS")
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{pat}@")
        subprocess.run(["git", "remote", "set-url", "origin", auth_url], check=True)
        subprocess.run(["git", "config", "user.email", "kaggle-bot@local"], check=True)
        subprocess.run(["git", "config", "user.name", "kaggle-bot"], check=True)
        print("git push auth configured")
    except Exception as e:
        print("⚠️  no GH_PAT_DELTA_MEM_TESTS secret — checkpoints will NOT be pushed:", e)
else:
    print("local run — assuming existing git remote auth")

## 6. Load or create run state

In [ ]:
import datetime

STAGE = os.environ.get("STAGE", "S3")    # set explicitly if running on S1/S2 locally
WALL_CLOCK_BUDGET = int(os.environ.get("WALL_CLOCK_BUDGET", 8 * 3600))

run_id = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}-{ENV}-{STAGE}"
run_state = state.load_or_create(run_id=run_id, stage=STAGE, budget_seconds=WALL_CLOCK_BUDGET)

print("run_id:", run_state.run_id)
print("stage:", run_state.stage)
print("completed cells so far:", run_state.completed_cells)
print(f"wall-clock budget: {run_state.wall_clock_budget_seconds}s ({run_state.wall_clock_budget_seconds/3600:.1f}h)")

## 7. Dispatch loop

For each pending cell in stage order:
1. Check `control/STATUS` and wall-clock — exit gracefully if either trips.
2. Run the cell (stubbed in this scaffold — real runners land in the implementation phase).
3. Write `results/{cell.id}.json`.
4. Commit + push the result back to the repo.
5. Update `state.json` and push it too.

In [ ]:
def commit_and_push(paths, message):
    subprocess.run(["git", "add", *paths], check=True)
    # Allow empty in case status file is unchanged
    r = subprocess.run(["git", "commit", "-m", message], capture_output=True, text=True)
    if r.returncode != 0 and "nothing to commit" not in r.stdout + r.stderr:
        raise RuntimeError(f"commit failed: {r.stderr}")
    push = subprocess.run(["git", "push"], capture_output=True, text=True)
    if push.returncode != 0:
        print("  ⚠️ push failed (continuing):", push.stderr.strip())

def run_cell_stub(cell):
    """Placeholder — real runners live in harness/runners/. Replace during impl phase."""
    return {
        "cell_id": cell.id,
        "title": cell.title,
        "status": "scaffold-stub",
        "note": "This is the scaffold. Implement runners under harness/runners/ to produce real metrics.",
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }

results_dir = WORKDIR / "results" / STAGE
results_dir.mkdir(parents=True, exist_ok=True)

for cell in cells.cells_for_stage(STAGE):
    if cell.id in run_state.completed_cells:
        print(f"✓ skipping {cell.id} ({cell.title}) — already done")
        continue
    if cell.blocked_by and cell.blocked_by not in run_state.completed_cells:
        print(f"⏸ skipping {cell.id} — blocked by {cell.blocked_by}")
        continue

    stop, reason = control.should_stop(run_state)
    if stop:
        print(f"🛑 stopping early: {reason}")
        break

    run_state.current_cell = cell.id
    run_state.save()
    print(f"▶ running cell {cell.id}: {cell.title}")

    result = run_cell_stub(cell)
    out_path = results_dir / f"cell-{cell.id}.json"
    out_path.write_text(json.dumps(result, indent=2))

    run_state.mark_completed(cell.id)
    commit_and_push(
        [str(out_path.relative_to(WORKDIR)), "state.json"],
        f"cell {cell.id}: {cell.title} [{run_state.stage}/{ENV}]",
    )
    print(f"  → wrote {out_path.relative_to(WORKDIR)} and pushed")

## 8. Render inline summary

In [ ]:
from IPython.display import Markdown, display

rows = []
for p in sorted((WORKDIR / "results" / STAGE).glob("cell-*.json")):
    d = json.loads(p.read_text())
    rows.append(f"| {d['cell_id']} | {d['title']} | {d['status']} |")

md = "## Results — " + STAGE + "\n\n| Cell | Title | Status |\n|---|---|---|\n" + "\n".join(rows)
display(Markdown(md))
(WORKDIR / "results" / "summary.md").write_text(md)
commit_and_push(["results/summary.md"], f"summary [{STAGE}/{ENV}]")

## 9. Self-terminate

Final commit pushed. After this cell, **also stop the Kaggle session manually** (top-right ⋯ → Stop session) so the kernel doesn't sit idle counting against your quota.

`os._exit(0)` short-circuits any later cells if someone hits Run All twice.

In [ ]:
print("All done. Stop the Kaggle session from the UI to release the GPU.")
# Uncomment for true self-terminate (kills the kernel):
# import os; os._exit(0)